In [3]:
# Install the relevant libraries required
!pip3 -q install -U langchain langchain-huggingface huggingface_hub python-dotenv --break-system-packages

## Prompt Engineering

Prompt Engineering is based off in-context learning, which lets LLMs draw on prior training experience by activating relevant patterns from examples in the prompt, without updating their weights.

In [16]:
# Load the libraries
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableSequence, RunnableLambda
from langchain_core.messages import HumanMessage, SystemMessage
from huggingface_hub import InferenceClient

# Load environment variables
load_dotenv()

# Disable warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

Create the shared function to invoke the prompts

In [21]:
def invoke(prompt, max_tokens=128):
    client = InferenceClient(
        model="zai-org/GLM-5",
        provider="novita",
        token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    )

    response = client.chat.completions.create(
        model="zai-org/GLM-5",
        messages=[
            {"role": "system", "content": "Do not show reasoning. Return only the final answer."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=0, # Lower means more deterministic and focused, and higher means more varied. This is for randomness.
        top_p=0.2,
        extra_body={
            "enable_thinking": False,
            "chat_template_kwargs": {"enable_thinking": False},
            "thinking": {"type": "disabled"},
        }, # Disable thinking

    )

    return response.choices[0].message.content

### Basic Prompt

Start with just a prompt to test out the code

In [16]:
invoke("Hello World!")

'Hello! How can I help you today'

###  📝 Exercise 1

Experiment with different basic prompts by changing the input phrase. Try these prompts and compare the different responses:

1. "The future of artificial intelligence is"
2. "Once upon a time in a distant galaxy"
3. "The benefits of sustainable energy include"

In [17]:
invoke("The future of artificial intelligence is")

'The future of artificial intelligence is poised to be transformative, likely integrating deeper'

In [18]:
invoke("Once upon a time in a distant galaxy")

''

In [19]:
invoke("The benefits of sustainable energy include")

'The benefits of sustainable energy include:\n\n*   **Reduced Greenhouse Gas Emissions:** Sustainable energy sources like wind, solar, and hydroelectric power produce little to no greenhouse gases during operation, helping to combat climate change and reduce global warming.\n*   **Improved Public Health:** By reducing reliance on fossil fuels, sustainable energy decreases air and water pollution. This leads to fewer respiratory illnesses, heart problems, and other health issues associated with smog'

### Zero-shot Prompt

Zero-shot prompting asks a model to do a task with no examples - e.g. tests whether a model can generalize to new tasks using only pre-trained knowledge without extra training.

Example: ask it to classify the statement “The Eiffel Tower is located in Berlin” as true or false, then run and print the response with default settings.

This shows how well it handles direct reasoning questions.


In [20]:
prompt = """Classify the following statement as true or false: 
            'The Eiffel Tower is located in Berlin.'

            Answer:
"""
response = invoke(prompt)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Classify the following statement as true or false: 
            'The Eiffel Tower is located in Berlin.'

            Answer:


response : False



###  📝 Exercise 2

Create zero-shot prompts for the following tasks:

Write a prompt that asks the model to classify a movie review as positive or negative.
Create a prompt that instructs the model to summarize a paragraph about climate change.
Design a prompt that asks the model to translate an English phrase to Spanish without showing any examples.

In [47]:
## Starter code: provide your solutions in the TODO parts

# 1. Prompt for Movie Review Classification
movie_review_prompt = """
Classify the following Netflix movie 96 Minutes as positive or negative based on the reviews. 
Answer: """ ## TODO

# 2. Prompt for Climate Change Paragraph Summarization
climate_change_prompt = "Summarize a paragraph on a topic about climate change, and be as concise as possible." ## TODO

# 3. Prompt for English to Spanish Translation
translation_prompt = "Translate this phrase 'May the Year of the Horse 2026 brings you proposerity, great success and wealth!' to Spanish" ## TODO

responses = {}
responses["movie_review"] = invoke(movie_review_prompt) ## TODO
responses["climate_change"] = invoke(climate_change_prompt) ## TODO
responses["translation"] = invoke(translation_prompt) ## TODO

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

=== MOVIE_REVIEW RESPONSE ===
Negative

=== CLIMATE_CHANGE RESPONSE ===
Climate change, driven primarily by human activities like burning fossil fuels, is causing significant global warming and environmental disruption. This results in rising sea levels, extreme weather events, and biodiversity loss, necessitating urgent global action to mitigate its severe impacts.

=== TRANSLATION RESPONSE ===
¡Que el Año del Caballo 2026 te traiga prosperidad, gran éxito y riqueza!



### One-shot Prompt

One-shot prompting gives one example, then asks for a similar output.
It helps the model follow the expected format/style with minimal guidance.

Example: show one English → French translation, then ask it to translate a new English sentence.

In [48]:
prompt = """Here is an example of translating a sentence from English to French:

            English: “How is the weather today?”
            French: “Comment est le temps aujourd'hui?”
            
            Now, translate the following sentence from English to French:
            
            English: “Where is the nearest supermarket?”            
"""

response = invoke(prompt)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Here is an example of translating a sentence from English to French:

            English: “How is the weather today?”
            French: “Comment est le temps aujourd'hui?”

            Now, translate the following sentence from English to French:

            English: “Where is the nearest supermarket?”            


response : French: “Où est le supermarché le plus proche?”



### 📝 Exercise 3
Develop one-shot prompts for these scenarios:

1. Create a prompt with one example of a formal email, then ask the model to write another formal email on a different topic.
2. Provide one example of converting a technical concept into a simple explanation, then ask the model to explain a different concept.
3. Give one example of extracting keywords from a sentence, then ask the model to extract keywords from a new sentence.

In [49]:
## Starter code: provide your solutions in the TODO parts

# 1. One-shot prompt for formal email writing
formal_email_prompt = """
Below is an example of a formal email for the next steps:

Dear Team, 

I hope this message finds you well. Please let me know a convenient time to discuss next steps.

Regards,
Marie

Now help me create a formal email to confirm the schedule for Monday morning 2pm:
""" ## TODO

# 2. One-shot prompt for simplifying technical concepts
technical_concept_prompt = """ 
Explain a concept simply, for example, on the topic of LLM: 
LLM is software trained on massive text, predicting words to answer questions and generate content.

Now, help me explain similarly for Quantum Compute:
""" ## TODO

# 3. Few-shot prompt for keyword extraction
keyword_extraction_prompt = """
Below is an example of some logs,

1. GET /api/v1/users HTTP/1.1 200 123 '-' 'curl/8.5.0' nginx-access
2. POST /login HTTP/1.1 401 89 'https://example.com/' 'Mozilla/5.0' nginx-access
3. GET /static/app.js HTTP/1.1 304 0 'https://example.com/dashboard' 'Chrome/122.0' nginx-access

When extracted it becomes like this

1. Method: GET, Path: /api/v1/users, Status: 200
2. Method: POST, Path: /login, Status: 401
3. Method: GET, Path: /static/app.js, Status: 304

Now do for these based on the extraction format,

4. PUT /api/v1/users/42 HTTP/1.1 204 0 'https://example.com/admin' 'PostmanRuntime/7.37.0' nginx-access
5. DELETE /api/v1/sessions/abc123 HTTP/1.1 200 56 'https://example.com/profile' 'Mozilla/5.0' nginx-access
6. HEAD /healthz HTTP/1.1 200 0 '-' 'kube-probe/1.30' nginx-access
""" ## TODO

responses = {}
responses["formal_email"] = invoke(formal_email_prompt) ## TODO
responses["technical_concept"] = invoke(technical_concept_prompt) ## TODO
responses["keyword_extraction"] = invoke(keyword_extraction_prompt) ## TODO

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

=== FORMAL_EMAIL RESPONSE ===
Subject: Schedule Confirmation - Monday Meeting

Dear Team,

I hope this message finds you well.

I am writing to confirm that the meeting is scheduled for Monday at 2:00 PM. Please let me know if this time works for you or if you have any conflicts.

Best regards,

[Your Name]

=== TECHNICAL_CONCEPT RESPONSE ===
Quantum computing is a machine that uses the physics of tiny particles to explore many possibilities at once, allowing it to solve specific complex problems much faster than regular computers.

=== KEYWORD_EXTRACTION RESPONSE ===
4. Method: PUT, Path: /api/v1/users/42, Status: 204
5. Method: DELETE, Path: /api/v1/sessions/abc123, Status: 200
6. Method: HEAD, Path: /healthz, Status: 200



### Few-shot Prompt

Few-shot prompting gives 2–5 examples before a task to set clearer patterns for format, style, and reasoning. It works better than one-shot for complex tasks.

Example: show labeled emotion samples, then classify a new sentence (e.g., fear from a scary movie statement).


In [ ]:
prompt = """Here are few examples of classifying emotions in statements:

Statement: 'I just won my first marathon!'
Emotion: Joy

Statement: 'I can't believe I lost my keys again.'
Emotion: Frustration

Statement: 'My best friend is moving to another country.'
Emotion: Sadness

Now, classify the emotion in the following statement:
Statement: 'That movie was so scary I had to cover my eyes.’
"""

response = invoke(prompt)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Here are few examples of classifying emotions in statements:

Statement: 'I just won my first marathon!'
Emotion: Joy

Statement: 'I can't believe I lost my keys again.'
Emotion: Frustration

Statement: 'My best friend is moving to another country.'
Emotion: Sadness

Now, classify the emotion in the following statement:
Statement: 'That movie was so scary I had to cover my eyes.’


response : Fear



### Chain-of-Thought (CoT) Prompt

Chain-of-thought prompting asks the model to reason step by step before answering. By encouraging the model to include explicit reasoning steps, it improves accuracy on multi-step tasks like math, logic, and decisions as it mimicks human-like problem-solving processes.

Unlike **One-Shot** prompting, it improves the results as it forces stepwise token (e.g. guessing or predicting the next token better during inference reading each token), thus reducing skipped logic as it has to build context between each steps.

Example: solve an apple-count problem by explicitly showing each calculation step.

In [7]:
prompt = """Consider the problem: 'A store had 22 apples. They sold 15 apples today and got a new delivery of 8 apples. 
How many apples are there now?’
Break down each step of your calculation
"""
response = invoke(prompt)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Consider the problem: 'A store had 22 apples. They sold 15 apples today and got a new delivery of 8 apples. 
How many apples are there now?’
Break down each step of your calculation


response : Here is the step-by-step calculation:

**Step 1: Subtract the apples sold.**
Start with the initial amount (22 apples) and subtract the number sold (15 apples).
$22 - 15 = 7$ apples remaining.

**Step 2: Add the new delivery.**
Take the remaining apples (7) and add the new delivery (8 apples).
$7 + 8 = 15$ apples.

**Final Answer:**
There are now **15** apples.



### 📝 Exercise 4

Create CoT prompts for these scenarios:

1. Write a prompt that asks the model to think through whether a student should study tonight or go to a movie with friends, considering their upcoming test in two days.
2. Write a prompt that instructs the model to explain the step-by-step process of making a peanut butter and jelly sandwich.

In [8]:
## Starter code: provide your solutions in the TODO parts

# 1. Prompt for decision-making process
decision_making_prompt = """ 
Consider this decision making problem if a student should study tonight or go to a movie with friends, considering their upcoming test in two days.
Explain line-by-line for your reasoning of choice.
""" ## TODO

# 2. Prompt for explaining a process
sandwich_making_prompt = """ 
Explain the step-by-step process of making peanut butter and jelly.
""" ## TODO

responses = {}
responses["decision_making"] = invoke(decision_making_prompt) ## TODO
responses["sandwich_making"] = invoke(sandwich_making_prompt) ## TODO

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

=== DECISION_MAKING RESPONSE ===
1. **Define the Objective:** The primary goal is to maximize the student's outcome, balancing immediate social enjoyment with future academic success.

2. **Analyze Option A (Go to a Movie):** This choice provides immediate gratification, stress relief, and social connection. However, it consumes time that could be used for preparation and may lead to increased anxiety or guilt regarding the upcoming test.

3. **Analyze Option B (Study Tonight):** This choice utilizes the available time to reinforce knowledge and build confidence for the test. It sacrifices immediate fun but invests in the long-term goal of achieving a good grade.

4. **

=== SANDWICH_MAKING RESPONSE ===
Here is the step-by-step process for making a peanut butter and jelly sandwich:

**Step 1: Gather Your Ingredients and Tools**
You will need two slices of bread, a jar of peanut butter, a jar of jelly or jam, a butter knife, and a plate.

**Step 2: Prepare the Bread**
Place the two slic

### Self-consistency

Self-consistency asks the model to generate multiple independent solutions, then pick the most consistent answer. This improves accuracy by comparing different reasoning paths and filtering weak ones.

Example: solve the sister-age problem using three separate calculations, then choose the consistent result.


In [11]:
prompt = """When I was 6, my sister was half of my age. Now I am 70, what age is my sister?
            Provide three independent calculations and explanations, then determine the most consistent result.
"""

response = invoke(prompt, max_tokens=512)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: When I was 6, my sister was half of my age. Now I am 70, what age is my sister?
            Provide three independent calculations and explanations, then determine the most consistent result.


response : **Calculation 1: Determining the Age Difference**
When you were 6, your sister was half your age, which is 3 years old ($6 \div 2 = 3$). This means your sister is 3 years younger than you ($6 - 3 = 3$). If you are now 70, your sister is still 3 years younger. Therefore, her age is $70 - 3 = 67$.

**Calculation 2: Calculating Years Elapsed**
At the starting point, you were 6 years old. Now you are 70 years old. The amount of time that has passed is 64 years ($70 - 6 = 64$). When you were 6, your sister was 3 years old. Adding the elapsed time to her starting age gives $3 + 64 = 67$.

**Calculation 3: Using the Age Ratio**
Initially, the ratio of your sister's age to yours was $1:2$ (she was 3 and you were 6). The fixed difference between your ages is 3 years. As you age, the ra

# Applications of prompting in different use cases

Use a repeatable LCEL (LangChain Expression Language) pattern: (1) define the task, (2) build a variable template, (3) convert to PromptTemplate, (4) chain input → prompt → LLM → parser with |, (5) then invoke with specific inputs.

This creates reusable, flexible NLP components with consistent results

In this template, define two parameters: `adjective` and `content`.

In [12]:
template = """Tell me a {adjective} joke about {content}."""

prompt = PromptTemplate.from_template(template)
prompt 

PromptTemplate(input_variables=['adjective', 'content'], input_types={}, partial_variables={}, template='Tell me a {adjective} joke about {content}.')

In [13]:
prompt.format(adjective="funny", content="chickens")

'Tell me a funny joke about chickens.'

### Creating a LCEL chain

In [25]:
llm = RunnableLambda(invoke)
format_prompt = lambda variables: prompt.format(**variables)

In [26]:
joke_chain = (
    RunnableLambda(lambda variables: prompt.format(**variables))
    | llm 
    | StrOutputParser()
)

# Run the chain
response = joke_chain.invoke({"adjective": "funny", "content": "chickens"}).strip()
print(response)

Why do chickens sit on their eggs?

Because they don't have chairs!


In [24]:
response = joke_chain.invoke({"adjective": "sad", "content": "fish"})
print(response)

Why are fish so easy to trick?
Because they fall for things hook, line, and sinker.


### Text Summarization

This summarization agent takes input text, runs it through an LCEL prompt-to-LLM chain, and returns a concise summary.
Store text in a variable to reuse the workflow with different content.

In [27]:
content = """
    The rapid advancement of technology in the 21st century has transformed various industries, including healthcare, education, and transportation. 
    Innovations such as artificial intelligence, machine learning, and the Internet of Things have revolutionized how we approach everyday tasks and complex problems. 
    For instance, AI-powered diagnostic tools are improving the accuracy and speed of medical diagnoses, while smart transportation systems are making cities more efficient and reducing traffic congestion. 
    Moreover, online learning platforms are making education more accessible to people around the world, breaking down geographical and financial barriers. 
    These technological developments are not only enhancing productivity but also contributing to a more interconnected and informed society.
"""

template = """Summarize the {content} in one sentence.
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
summarize_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Run the chain
summary = summarize_chain.invoke({"content": content})
print(summary)

Rapid 21st-century technological advancements, such as AI and IoT, have revolutionized industries like healthcare and education, enhancing productivity and fostering a more interconnected society.


### Question Answering

This LCEL Q&A agent answers questions using provided context.
To reduce speculation, it is instructed to reply “Unsure about the answer” when uncertain.
The chain takes both context and question, then sends them through the template to the LLM.

In [28]:
content = """
    The solar system consists of the Sun, eight planets, their moons, dwarf planets, and smaller objects like asteroids and comets. 
    The inner planets—Mercury, Venus, Earth, and Mars—are rocky and solid. 
    The outer planets—Jupiter, Saturn, Uranus, and Neptune—are much larger and gaseous.
"""

question = "Which planets in the solar system are rocky and solid?"

template = """
    Answer the {question} based on the {content}.
    Respond "Unsure about answer" if not sure about the answer.
    
    Answer:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
qa_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Run the chain
answer = qa_chain.invoke({"question": question, "content": content})
print(answer)

The inner planets—Mercury, Venus, Earth, and Mars—are rocky and solid.


### Text Classification

This LCEL text classification agent uses zero-shot learning to classify text into predefined categories without example training.

The chain accepts both the input text and category list.

In [29]:
text = """
    The concert last night was an exhilarating experience with outstanding performances by all artists.
"""

categories = "Entertainment, Food and Dining, Technology, Literature, Music."

template = """
    Classify the {text} into one of the {categories}.
    
    Category:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
classification_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Run the chain
category = classification_chain.invoke({"text": text, "categories": categories})
print(category)

Music


### Code Generation

This LCEL SQL generation agent converts natural language requirements into properly formatted executable SQL queries.

In [31]:
description = """
    Retrieve the names and email addresses of all customers from the 'customers' table who have made a purchase in the last 30 days. 
    The table 'purchases' contains a column 'purchase_date'
"""

template = """
    Generate an SQL query based on the {description}
    
    SQL Query:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
sql_generation_chain = (
    RunnableLambda(format_prompt) 
    | llm 
    | StrOutputParser()
)

# Run the chain
sql_query = sql_generation_chain.invoke({"description": description})
print(sql_query)

SELECT DISTINCT c.name, c.email
FROM customers c
JOIN purchases p ON c.id = p.customer_id
WHERE p.purchase_date >= DATE('now', '-30 days');


### Role-Playing

This LCEL Role-Playing agent allows parameterizing role, tone, and question so the LLM follows task-specific behavior without rewriting prompts.
This enables fast role-switching and reusable chatbot design across contexts.

Example: set it as a game master for RPG guidance, run in a loop, and exit with quit, exit, or bye.


In [33]:
role = """
    Dungeon & Dragons game master
"""

tone = "engaging and immersive"

template = """
    You are an expert {role}. I have this question {question}. I would like our conversation to be {tone}.
    
    Answer:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
roleplay_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Create an interactive chat loop
while True:
    query = input("Question: ")
    
    if query.lower() in ["quit", "exit", "bye"]:
        print("Answer: Goodbye!")
        break
        
    response = roleplay_chain.invoke({"role": role, "question": query, "tone": tone})
    print("Answer: ", response)

Answer:  *The ancient tome falls open before you, its pages crackling with arcane energy. A spectral figure materializes in the flickering torchlight—your Dungeon Master, cloaked in shadows and starlight.*

"Ah, a seeker of knowledge approaches the table of destiny! Welcome, brave adventurer. Your query has been whispered across the planes, yet the scroll you present bears no inscription..."

*The ghostly figure gestures to the empty space where your question should be.*

"**Speak your question, and I shall conjure an answer worthy of the gods themselves.** Whether you seek rules clarification, lore from the Forgotten Realms, character build
Answer: Goodbye!


### 📝 Exercise 5

##### Create an LCEL Chain with Custom Formatting

In this exercise, you'll create your own LCEL chain that uses prompt templates to build a custom application.

##### Task:
Create a product review analyzer that can:

Identify the sentiment (positive, negative, or neutral).
Extract mentioned product features.
Provide a one-sentence summary of the review.

##### Steps:

Create a prompt template with placeholders for the review text.
Build an LCEL chain that formats your prompt properly.
Process the sample reviews and display the results.
Try modifying the chain to change the output format.

##### Sample input:

```
reviews = [
    "I love this smartphone! The camera quality is exceptional and the battery lasts all day. The only downside is that it heats up a bit during gaming.",
    "This laptop is terrible. It's slow, crashes frequently, and the keyboard stopped working after just two months. Customer service was unhelpful."
]
```

In [38]:
# TODO: Initialize your LLM
llm = RunnableLambda(invoke)

# Here is an example template you can use
template = """
Analyze the following product review:
"{review}"

Provide your analysis in the following format:
- Sentiment: (positive, negative, or neutral)
- Key Features Mentioned: (list the product features mentioned)
- Summary: (one-sentence summary)
"""

# TODO: Create your prompt template
product_review_prompt = PromptTemplate.from_template(template) ## TODO

# TODO: Create a formatting function
def format_review_prompt(variables):
    # Your code here
    return product_review_prompt.format(**variables)

# TODO: Build your LCEL chain
review_analysis_chain = (
    # Your code here
    RunnableLambda(format_review_prompt)
    | llm 
    | StrOutputParser()
)

# Example reviews to process
reviews = [
    "I love this smartphone! The camera quality is exceptional and the battery lasts all day. The only downside is that it heats up a bit during gaming.",
    "This laptop is terrible. It's slow, crashes frequently, and the keyboard stopped working after just two months. Customer service was unhelpful."
]

# TODO: Process the reviews
for review in reviews:
    # Your code here
    response = review_analysis_chain.invoke({"review": review})
    print("=" * 8 + " Review " + "=" * 8)
    print(review)
    print(response)

======== Review ========
I love this smartphone! The camera quality is exceptional and the battery lasts all day. The only downside is that it heats up a bit during gaming.
- Sentiment: positive
- Key Features Mentioned: Camera quality, battery life, heating during gaming
- Summary: The reviewer praises the smartphone's camera and battery but notes that it tends to heat up while gaming.
======== Review ========
This laptop is terrible. It's slow, crashes frequently, and the keyboard stopped working after just two months. Customer service was unhelpful.
- Sentiment: negative
- Key Features Mentioned: Performance (speed), Stability (crashes), Keyboard, Customer Service
- Summary: The reviewer strongly criticizes the laptop for its poor performance, hardware failure, and unhelpful customer support.
